## Using a Triton Inference Server

[Triton Inference Server](https://developer.nvidia.com/triton-inference-server) is an open-source project by NVIDIA for high-performance ML model deployment. In this section, we will practice deploying models using Triton; after you have finished, you should be able to:

-   serve a model using Triton Inference Server with Python backend
-   understand how request queuing affects latency under load
-   compare constant vs. variable (Poisson) request arrival patterns
-   understand the effect of batch size on throughput and latency
-   use dynamic batching to improve throughput and reduce queuing delay
-   scale your model to run on multiple GPUs, and/or with multiple instances on the same GPU
-   benchmark the Triton service, and recognize indications of potential problems
-   and use optimized backends

### Anatomy of a Triton model with Python backend

To start, run

``` bash
# runs on node-serve-system
mkdir ~/serve-system-chi/models/
cp -r ~/serve-system-chi/models_staging/food_classifier ~/serve-system-chi/models/
```

to copy [our first configuration](https://github.com/teaching-on-testbeds/serve-system-chi/tree/main/models_staging/food_classifier) into the directory from which Triton will load models.

Our initial implementation serves our food image classifier using PyTorch. Here’s how it works.

In the [Dockerfile](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/docker/Dockerfile.triton), the Triton server is started with the command

``` bash
tritonserver --model-repository=/models
```

where the `/models` directry is organized as follows:

    models/
    └── food_classifier
        ├── 1
        │   ├── food11.pth
        │   └── model.py
        └── config.pbtxt

It includes:

-   a top-level directory whose name is the “model name”
-   a configuration file `config.pbtxt` inside that directory. We’ll look at that shortly.
-   and a subdirectory for each model version. We have model version 1, so we have a subdirectory 1. Inside this directory is a `model.py`, which describes how the model will run.

Let’s [look at the configuration file first](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/models_staging/food_classifier/config.pbtxt). Here are the contents of `config.pbtxt`:

    name: "food_classifier"
    backend: "python"
    max_batch_size: 16
    input [
      {
        name: "INPUT_IMAGE"
        data_type: TYPE_STRING
        dims: [1]
      }
    ]
    output [
      {
        name: "FOOD_LABEL"
        data_type: TYPE_STRING
        dims: [1]
      },
      {
        name: "PROBABILITY"
        data_type: TYPE_FP32
        dims: [1]
      }
    ]
      instance_group [
        {
          count: 1
          kind: KIND_GPU
          gpus: [ 0 ]
        }
    ]

We have defined:

-   a `name`, which must match the directory name
-   a `backend` - we are using the basic [Python backend](https://github.com/triton-inference-server/python_backend). This is a highly flexible backend which allows us to define how our model will run by providing Python code in a `model.py` file.
-   a `max_batch_size` - we have set it to 16, but generally you would set this according to the GPU memory available
-   the `name`, `data_type`, and `dims` (dimensions) of each `input` to the model
-   the `name`, `data_type`, and `dims` (dimensions) of each `output` from the model
-   an `instance_group` with the `count` (number of copies of the model that we want to serve) and details of the device we want to serve it on (we will serve it on GPU 0). Note that to run the model on CPU instead, we could have used

<!-- -->

      instance_group [
        {
          count: 1
          kind: KIND_CPU
        }
      ]

Next, let’s [look at `model.py`](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/models_staging/food_classifier/1/model.py). For a Triton model with Python backend, the `model.py` must define a class named `TritonPythonModel` with at least an `initialize` and `execute` method. Ours has:

-   An `initialize` method to load the model, move it to the device specified in the `args` passed from the Triton server, and put it in inference mode. This will run as soon as Triton starts and loads models from the directory passed to it:

``` python
def initialize(self, args):
        model_dir = os.path.dirname(__file__)
        model_path = os.path.join(model_dir, "food11.pth")
        
        # From args, get info about what device the model is supposed to be on
        instance_kind = args.get("model_instance_kind", "cpu").lower()
        if instance_kind == "gpu":
            device_id = int(args.get("model_instance_device_id", 0))
            torch.cuda.set_device(device_id)
            self.device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device('cpu')

        self.model = torch.load(model_path, map_location=self.device, weights_only=False)
        self.model.to(self.device)
        self.model.eval()

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225]),
        ])
        self.classes = np.array([
            "Bread", "Dairy product", "Dessert", "Egg", "Fried food",
            "Meat", "Noodles/Pasta", "Rice", "Seafood", "Soup",
            "Vegetable/Fruit"
        ])
```

-   A `preprocess` method, which will run on each input image that is passed:

``` python
def preprocess(self, image_data):
    if isinstance(image_data, str):
        image_data = base64.b64decode(image_data)

    if isinstance(image_data, bytes):
        image_data = image_data.decode("utf-8")
        image_data = base64.b64decode(image_data)

    image = Image.open(io.BytesIO(image_data)).convert('RGB')

    img_tensor = self.transform(image).unsqueeze(0)
    return img_tensor
```

-   and an `execute`, which will apply to batches of requests sent to this model:

``` python
def execute(self, requests):
    # Gather inputs from all requests
    batched_inputs = []
    for request in requests:
        in_tensor = pb_utils.get_input_tensor_by_name(request, "INPUT_IMAGE")
        input_data_array = in_tensor.as_numpy()  # each assumed to be shape [1]
        # Preprocess each input (resulting in a tensor of shape [1, C, H, W])
        batched_inputs.append(self.preprocess(input_data_array[0, 0]))
    
    # Combine inputs along the batch dimension
    batched_tensor = torch.cat(batched_inputs, dim=0).to(self.device)
    print("BatchSize: ", len(batched_inputs))
    # Run inference once on the full batch
    with torch.no_grad():
        outputs = self.model(batched_tensor)
    
    # Process the outputs and split them for each request
    responses = []
    for i, request in enumerate(requests):
        output = outputs[i:i+1]  # select the i-th output
        prob, predicted_class = torch.max(output, 1)
        predicted_label = self.classes[predicted_class.item()]
        probability = torch.sigmoid(prob).item()
        
        # Create numpy arrays with shape [1, 1] for consistency.
        out_label_np = np.array([[predicted_label]], dtype=object)
        out_prob_np = np.array([[probability]], dtype=np.float32)
        
        out_tensor_label = pb_utils.Tensor("FOOD_LABEL", out_label_np)
        out_tensor_prob = pb_utils.Tensor("PROBABILITY", out_prob_np)
        
        inference_response = pb_utils.InferenceResponse(
            output_tensors=[out_tensor_label, out_tensor_prob])
        responses.append(inference_response)
    
    return responses
```

Finally, now that we understand how the server works, let’s [look at how the Flask app sends requests to it](https://github.com/teaching-on-testbeds/gourmetgram/blob/triton/app.py). Inside the Flask app, we now have a function which is called whenever there is a new image uploaded to `predict` or `test`, which sends the image to the Triton server:

``` python
def request_triton(image_path):
    try:
        # Connect to Triton server
        triton_client = httpclient.InferenceServerClient(url=TRITON_SERVER_URL)

        # Prepare inputs and outputs
        with open(image_path, 'rb') as f:
            image_bytes = f.read()

        inputs = []
        inputs.append(httpclient.InferInput("INPUT_IMAGE", [1, 1], "BYTES"))

        encoded_str =  base64.b64encode(image_bytes).decode("utf-8")
        input_data = np.array([[encoded_str]], dtype=object)
        inputs[0].set_data_from_numpy(input_data)

        outputs = []
        outputs.append(httpclient.InferRequestedOutput("FOOD_LABEL", binary_data=False))
        outputs.append(httpclient.InferRequestedOutput("PROBABILITY", binary_data=False))

        # Run inference
        results = triton_client.infer(model_name=FOOD11_MODEL_NAME, inputs=inputs, outputs=outputs)

        predicted_class = results.as_numpy("FOOD_LABEL")[0,0]
        probability = results.as_numpy("PROBABILITY")[0,0]

        return predicted_class, probability

    except Exception as e:
        print(f"Error during inference: {e}")  
        return None, None  
```

### Bring up containers

To start, run

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up -d
```

This uses a [Docker Compose configuration](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/docker/docker-compose-triton.yaml) to bring up three containers:

-   one container with NVIDIA Triton Server, with the host’s GPUs passed to the container, and with the `models` directory (containing the model and its configuration) passed as a bind mount
-   one container that hosts the Flask app, which will serve the user interface and send inference requests to the Triton server
-   one Jupyter container with the Triton client installed, for us to conduct a performance evaluation of the Triton server

Watch the logs from the Triton server as it starts up:

``` bash
# runs on node-serve-system
docker logs triton_server -f
```

Once the Triton server starts up, you should see something like

    +--------------------------+---------+--------+
    | Model                    | Version | Status |
    +--------------------------+---------+--------+
    | food_classifier | 1       | READY  |
    +--------------------------+---------+--------+

and then some additional output. Near the end, you will see

    "Started GRPCInferenceService at 0.0.0.0:8001"
    "Started HTTPService at 0.0.0.0:8000"
    "Started Metrics Service at 0.0.0.0:8002"

(and then some messages about not getting GPU power consumption, which is fine and not a concern.)

You can use Ctrl+C to stop watching the logs once you see this output.

Let’s test this service. In a browser, run

    http://A.B.C.D

but substitute the floating IP assigned to your instance, to access the Flask app. Upload an image and press “Submit” to get its class label.

Finally, check the logs of the Jupyter container:

``` bash
# runs on node-serve-system
docker logs jupyter
```

and look for a line like

    http://127.0.0.1:8888/lab?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Paste this into a browser tab, but in place of 127.0.0.1, substitute the floating IP assigned to your instance, to open the Jupyter notebook interface that is running *on your compute instance*.

Then, in the file browser on the left side, open the “work” directory and then click on the `triton_v2.ipynb` notebook to continue.

Meanwhile, on the host, run

``` bash
# runs on node-serve-system
nvtop
```

to monitor GPU usage - we will refer back to this a few times as we run through the rest of this notebook.

### Baseline - Single Request Performance

The Triton client comes with a performance analyzer, which we can use to send requests to the server and measure performance. It reports:

-   **Throughput** - inferences per second
-   **Latency breakdown** - total request latency split into:
    -   `queue` - time spent waiting in line
    -   `compute infer` - actual inference time on the GPU
    -   `compute input` / `compute output` - input and output processing overhead

Let us start by establishing a baseline with a single client sending one request at a time:

In [2]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 944
    Throughput: 52.2649 infer/sec
    Avg latency: 19043 usec (standard deviation 392 usec)
    p50 latency: 19011 usec
    p90 latency: 19135 usec
    p95 latency: 19170 usec
    p99 latency: 19378 usec
    Avg HTTP time: 19037 usec (send/recv 175 usec + response wait 18862 usec)
  Server: 
    Inference count: 944
    Execution count: 944
    Successful request count: 944
    Avg request latency: 18432 usec (overhead 2 usec + queue 21 usec + compute input 45 usec + compute infer 18314 usec + compute output 50 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 52.2649 infer/sec, latency 19043 usec


<!--

    Avg request latency: 18689 usec (overhead 2 usec + queue 22 usec + compute input 44 usec + compute infer 18570 usec + compute output 49 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 51.549 infer/sec, latency 19311 usec

-->

Make a note of the following from the output:

-   **Total average request latency** - this is your baseline latency for a single request
-   **`queue` delay** - should be negligible (near zero), since there is only one client and no contention
-   **`compute infer`** - the actual GPU inference time
-   **Throughput** (inferences/second)

With a single client sending one request at a time, there is essentially no queuing delay. The total latency is dominated by inference computation. This gives us our baseline. Keep these numbers in mind - we will compare all subsequent experiments against them.

But what happens when multiple users send requests at the same time?

### The Queuing Problem - What Happens Under Load

In production, your model does not serve one user at a time. Multiple requests arrive concurrently. Let us see what happens to our service as load increases.

In the baseline test above, a single client sends continuous requests to the server - each time a response is returned, a new request is generated. Now, let us configure **8** concurrent clients, each sending continuous requests - as soon as any client gets a response, it sends a new request:

In [3]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 976
    Throughput: 54.0293 infer/sec
    Avg latency: 147348 usec (standard deviation 6081 usec)
    p50 latency: 147805 usec
    p90 latency: 148132 usec
    p95 latency: 148389 usec
    p99 latency: 150417 usec
    Avg HTTP time: 147342 usec (send/recv 182 usec + response wait 147160 usec)
  Server: 
    Inference count: 976
    Execution count: 976
    Successful request count: 976
    Avg request latency: 146695 usec (overhead 2 usec + queue 128245 usec + compute input 48 usec + compute infer 18348 usec + compute output 50 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 54.0293 infer/sec, latency

While the inference time (`compute infer`) remains low, the overall system latency is high because of `queue` delay. Only one sample is processed at a time, and other samples have to wait in a queue for their turn. Since there are 8 concurrent clients sending continuous requests, the delay is approximately 8x the inference delay.

Notice that throughput barely changed from the baseline\! The server is still processing one request at a time at the same rate - the extra clients are just waiting in line.

With more concurrent requests, the queuing delay grows even larger:

In [4]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 16

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 16
  Client: 
    Request count: 978
    Throughput: 54.0517 infer/sec
    Avg latency: 293586 usec (standard deviation 21245 usec)
    p50 latency: 289599 usec
    p90 latency: 317371 usec
    p95 latency: 318271 usec
    p99 latency: 320508 usec
    Avg HTTP time: 293580 usec (send/recv 522 usec + response wait 293058 usec)
  Server: 
    Inference count: 978
    Execution count: 978
    Successful request count: 978
    Avg request latency: 292609 usec (overhead 2 usec + queue 274170 usec + compute input 48 usec + compute infer 18339 usec + compute output 49 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 16, throughput: 54.0517 infer/sec, late

Doubling concurrency doubled the queue delay, but throughput stayed flat. The server processes one request at a time at the same rate regardless of how many clients are waiting. This is a classic queuing problem.

Although the delay is large, it is not because of inadequate compute - if you check the `nvtop` display on the host while the test above is running, you will note low GPU utilization\! Take a screenshot of the `nvtop` output when this test is running.

#### Constant vs. Poisson request patterns

The concurrency-based tests above push the server as hard as possible. But in the real world, requests do not arrive at a perfectly constant rate - they arrive in bursts. A more realistic model of request arrivals is a [Poisson process](https://en.wikipedia.org/wiki/Poisson_point_process), where the time between requests is random (exponentially distributed) with some average rate.

The `perf_analyzer` tool supports this with two flags:

-   `--request-rate-range N` - send requests at an average rate of N requests per second
-   `--request-distribution constant|poisson` - either evenly spaced (`constant`) or randomly spaced (`poisson`)

Let us compare these two patterns. First, let us send requests at **30 requests per second** with constant spacing.

In [10]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 30 --request-distribution constant 

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using synchronous calls for inference

Request Rate: 30 inference requests per second
  Client: 
    Request count: 540
    Throughput: 29.9355 infer/sec
    Avg latency: 21349 usec (standard deviation 858 usec)
    p50 latency: 21362 usec
    p90 latency: 21715 usec
    p95 latency: 21803 usec
    p99 latency: 24245 usec
    Avg HTTP time: 21343 usec (send/recv 173 usec + response wait 21170 usec)
  Server: 
    Inference count: 540
    Execution count: 540
    Successful request count: 540
    Avg request latency: 20723 usec (overhead 2 usec + queue 20 usec + compute input 53 usec + compute infer 20599 usec + compute output 48 usec)
Inferences/Second vs. Client Average Batch 

At 30 requests per second with constant spacing, the server can keep up easily. Requests arrive at regular intervals and each finishes well before the next arrives, so queue delay should be minimal.

Now let us increase to **50 requests per second**

In [11]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 50 --request-distribution constant 

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using synchronous calls for inference

Request Rate: 50 inference requests per second
  Client: 
    Request count: 902
    Throughput: 49.939 infer/sec
    Avg latency: 18557 usec (standard deviation 854 usec)
    p50 latency: 18412 usec
    p90 latency: 18926 usec
    p95 latency: 19102 usec
    p99 latency: 20393 usec
    Avg HTTP time: 18551 usec (send/recv 207 usec + response wait 18344 usec)
  Server: 
    Inference count: 902
    Execution count: 902
    Successful request count: 902
    Avg request latency: 17897 usec (overhead 2 usec + queue 69 usec + compute input 47 usec + compute infer 17730 usec + compute output 47 usec)
Inferences/Second vs. Client Average Batch L

Look at compute infer: for 30 req/sec vs 50 req/sec. The actual GPU inference is faster at the higher rate.

This is due to GPU clock scaling. At 30 req/sec, requests arrive every ~33ms with gaps between them, so the GPU clocks down between requests. At 50 req/sec, requests arrive every ~20ms, keeping the GPU in a higher clock state, which makes each individual inference faster.

The queue delay is slightly higher at 50 (69 vs 20 usec), which is expected. But the GPU warmth effect more than compensates.

At 50 requests per second with constant spacing, queue delay should be noticeable. We should also see a slightly higher GPU utilization.

Now, let us switch to **Poisson arrivals** at the same rate of 50 requests per second:

In [12]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 50 --request-distribution poisson

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 50 inference requests per second
  Client: 
    Request count: 910
    Avg send request rate: 50.48 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 46.59% of the requests were delayed. 
    Throughput: 50.43 infer/sec
    Avg latency: 56002 usec (standard deviation 19949 usec)
    p50 latency: 68186 usec
    p90 latency: 72690 usec
    p95 latency: 73380 usec
    p99 latency: 73783 usec
    Avg HTTP time: 55996 usec (send/recv 202 usec + response wait 55794 usec)
  Server: 
    Inference count: 910
    Execution count: 910
    Successful request count: 910
    Avg request latency: 

Compare this directly to the constant-rate test at 50 req/sec above. With Poisson arrivals at the same average rate, requests sometimes arrive in bursts and sometimes with gaps. The bursts cause queue buildup, leading to significantly higher queue delay even though the average rate is the same.

Let us also try Poisson arrivals at the lower rate of **30 requests per second**:

In [13]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 30 --request-distribution poisson

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 30 inference requests per second
  Client: 
    Request count: 516
    Throughput: 28.5917 infer/sec
    Avg latency: 34462 usec (standard deviation 18566 usec)
    p50 latency: 27811 usec
    p90 latency: 61454 usec
    p95 latency: 75282 usec
    p99 latency: 94545 usec
    Avg HTTP time: 34442 usec (send/recv 216 usec + response wait 34226 usec)
  Server: 
    Inference count: 516
    Execution count: 516
    Successful request count: 516
    Avg request latency: 33760 usec (overhead 2 usec + queue 14291 usec + compute input 63 usec + compute infer 19350 usec + compute output 53 usec)
Inferences/Second vs. Client Average Batch Latenc

Even at 30 requests per second, Poisson arrivals show more queue delay than constant arrivals at the same rate. The burstiness creates temporary overloads even when the average load is manageable.

This is the fundamental insight from queuing theory: **variable arrivals at near-capacity rates cause disproportionately higher delays than constant arrivals.** Real-world traffic is bursty, so we must design our serving system to handle Poisson-like patterns, not just constant ones.

We have established that queuing is the bottleneck, and that real-world bursty traffic makes it worse. Can we process requests more efficiently? Let us explore the effect of batching.

### Effect of Client-Side Batch Size

We have seen that queuing is the bottleneck, and that real-world bursty traffic makes it worse. One way to improve throughput is **batching** - processing multiple images in a single pass through the GPU.

There are two ways to achieve batching:

-   **Client-side (simple) batching**: the client bundles multiple images into a single request using the `-b` flag in `perf_analyzer`. The server receives one request containing multiple images.
-   **Server-side (dynamic) batching**: each client sends a single image, but the server automatically groups multiple waiting requests into a batch before sending them to the GPU.

Triton does not offer a "simple" server-side batching mode (e.g., "wait for exactly 8 requests then process") because it would be rigid and wasteful - what if only 5 requests arrive and traffic stops? Those 5 would wait forever. Instead, Triton offers **dynamic batching**, which we will explore in the next section.

For now, let us use client-side batching to understand **why batching helps** - how batch size affects throughput and latency. This will build intuition for why dynamic batching is so effective.

#### Batch size sweep with concurrency 1

Let us start with a single client. First, the baseline with no batching (batch size 1):

In [15]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 961
    Throughput: 53.2017 infer/sec
    Avg latency: 18709 usec (standard deviation 773 usec)
    p50 latency: 18631 usec
    p90 latency: 18858 usec
    p95 latency: 19002 usec
    p99 latency: 19370 usec
    Avg HTTP time: 18704 usec (send/recv 174 usec + response wait 18530 usec)
  Server: 
    Inference count: 961
    Execution count: 961
    Successful request count: 961
    Avg request latency: 18098 usec (overhead 2 usec + queue 28 usec + compute input 57 usec + compute infer 17954 usec + compute output 57 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 53.2017 infer/sec, latency 18709 usec


Now let us increase the batch size - the client sends 4, 8, then 16 images per request:

In [16]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 4 --concurrency-range 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 4
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 931
    Throughput: 206.078 infer/sec
    Avg latency: 19318 usec (standard deviation 182 usec)
    p50 latency: 19309 usec
    p90 latency: 19488 usec
    p95 latency: 19537 usec
    p99 latency: 19718 usec
    Avg HTTP time: 19312 usec (send/recv 213 usec + response wait 19099 usec)
  Server: 
    Inference count: 3724
    Execution count: 931
    Successful request count: 931
    Avg request latency: 18178 usec (overhead 2 usec + queue 20 usec + compute input 115 usec + compute infer 17989 usec + compute output 50 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 206.078 infer/sec, latency 19318 usec

In [17]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 8 --concurrency-range 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 8
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 892
    Throughput: 395.155 infer/sec
    Avg latency: 20152 usec (standard deviation 389 usec)
    p50 latency: 20121 usec
    p90 latency: 20546 usec
    p95 latency: 20657 usec
    p99 latency: 20926 usec
    Avg HTTP time: 20147 usec (send/recv 278 usec + response wait 19869 usec)
  Server: 
    Inference count: 7136
    Execution count: 892
    Successful request count: 892
    Avg request latency: 18349 usec (overhead 2 usec + queue 20 usec + compute input 203 usec + compute infer 18072 usec + compute output 51 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 395.155 infer/sec, latency 20152 usec

In [18]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 16 --concurrency-range 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 16
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 814
    Throughput: 721.348 infer/sec
    Avg latency: 22107 usec (standard deviation 374 usec)
    p50 latency: 22068 usec
    p90 latency: 22415 usec
    p95 latency: 22478 usec
    p99 latency: 22786 usec
    Avg HTTP time: 22102 usec (send/recv 374 usec + response wait 21728 usec)
  Server: 
    Inference count: 13024
    Execution count: 814
    Successful request count: 814
    Avg request latency: 18755 usec (overhead 3 usec + queue 21 usec + compute input 236 usec + compute infer 18444 usec + compute output 51 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 721.348 infer/sec, latency 22107 us

With a single client, there is no contention - queue delay should be near zero for all batch sizes. But notice how throughput scales dramatically with batch size while per-batch latency barely increases. The GPU processes a batch of 16 in barely more time than a single image.

#### Batch size sweep with concurrency 8

Now let us repeat the same experiment with 8 concurrent clients. First, the baseline with no batching - recall from Section 5 that this had very high queue delay:

In [19]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 1004
    Throughput: 55.5641 infer/sec
    Avg latency: 143373 usec (standard deviation 5659 usec)
    p50 latency: 143430 usec
    p90 latency: 144823 usec
    p95 latency: 145184 usec
    p99 latency: 151256 usec
    Avg HTTP time: 143367 usec (send/recv 177 usec + response wait 143190 usec)
  Server: 
    Inference count: 1004
    Execution count: 1004
    Successful request count: 1004
    Avg request latency: 142758 usec (overhead 2 usec + queue 124808 usec + compute input 58 usec + compute infer 17840 usec + compute output 48 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 55.5641 infer/sec, lat

In [25]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 32

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 32
  Client: 
    Request count: 1007
    Throughput: 55.6124 infer/sec
    Avg latency: 566011 usec (standard deviation 54196 usec)
    p50 latency: 573885 usec
    p90 latency: 576223 usec
    p95 latency: 584219 usec
    p99 latency: 586310 usec
    Avg HTTP time: 566004 usec (send/recv 785 usec + response wait 565219 usec)
  Server: 
    Inference count: 1007
    Execution count: 1007
    Successful request count: 1007
    Avg request latency: 564717 usec (overhead 3 usec + queue 546809 usec + compute input 60 usec + compute infer 17792 usec + compute output 52 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 32, throughput: 55.6124 infer/sec, 

Now that we have the results for what happens when we have 32 concurrent requests being sent. We can divide them into a batch of 4 from the client side and see what happens. We should expect higher throughtput and a significant drop in queueing delay. 

In [20]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 4 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 4
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 994
    Throughput: 220.045 infer/sec
    Avg latency: 144725 usec (standard deviation 6806 usec)
    p50 latency: 144467 usec
    p90 latency: 148394 usec
    p95 latency: 148804 usec
    p99 latency: 149169 usec
    Avg HTTP time: 144719 usec (send/recv 221 usec + response wait 144498 usec)
  Server: 
    Inference count: 3976
    Execution count: 994
    Successful request count: 994
    Avg request latency: 143539 usec (overhead 2 usec + queue 125419 usec + compute input 123 usec + compute infer 17945 usec + compute output 49 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 220.045 infer/sec, laten

Lets see what happens if we further increase the batch size:

In [26]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 8 --concurrency-range 4

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 8
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 4
  Client: 
    Request count: 988
    Throughput: 437.384 infer/sec
    Avg latency: 72920 usec (standard deviation 2013 usec)
    p50 latency: 72853 usec
    p90 latency: 73345 usec
    p95 latency: 73537 usec
    p99 latency: 73988 usec
    Avg HTTP time: 72914 usec (send/recv 312 usec + response wait 72602 usec)
  Server: 
    Inference count: 7904
    Execution count: 988
    Successful request count: 988
    Avg request latency: 70938 usec (overhead 2 usec + queue 52714 usec + compute input 208 usec + compute infer 17963 usec + compute output 51 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 4, throughput: 437.384 infer/sec, latency 72920 

We can clearly see that the throughput and queueing delays are much better. This way we can process a lot of requests by batching

In [21]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 8 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 8
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 991
    Throughput: 438.504 infer/sec
    Avg latency: 145344 usec (standard deviation 6440 usec)
    p50 latency: 145776 usec
    p90 latency: 146470 usec
    p95 latency: 146719 usec
    p99 latency: 146978 usec
    Avg HTTP time: 145338 usec (send/recv 275 usec + response wait 145063 usec)
  Server: 
    Inference count: 7928
    Execution count: 991
    Successful request count: 991
    Avg request latency: 143276 usec (overhead 2 usec + queue 125079 usec + compute input 209 usec + compute infer 17934 usec + compute output 50 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 438.504 infer/sec, laten

In [24]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 16 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 16
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 962
    Throughput: 851.584 infer/sec
    Avg latency: 149673 usec (standard deviation 6621 usec)
    p50 latency: 149801 usec
    p90 latency: 152288 usec
    p95 latency: 153028 usec
    p99 latency: 153315 usec
    Avg HTTP time: 149666 usec (send/recv 414 usec + response wait 149252 usec)
  Server: 
    Inference count: 15392
    Execution count: 962
    Successful request count: 962
    Avg request latency: 146252 usec (overhead 3 usec + queue 127513 usec + compute input 259 usec + compute infer 18425 usec + compute output 52 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 851.584 infer/sec, lat

Compare these results carefully. You should notice:

-   **Throughput scales with batch size** - with batch size 16, you get roughly 16x the throughput of batch size 1, because the GPU processes 16 images in nearly the same time as 1.
-   **Queue delay stays the same** - the server is able to increase throughput and reduce queueing delay if we try to keep the essential request rate same. But we still are doing the same thing, processing one requests at a time. If number of requests increases then we still have higher and higher queueing delays.
-   **Total latency barely changes** - from the perspective of each client, the wait is the same.

Client-side batching improves **throughput** (more images per second) but does **not** solve the queuing problem. The fundamental issue remains: requests are processed one at a time, and the queue delay is unchanged.

Furthermore, client-side batching requires the client to have multiple images ready before sending a request. In a real-world scenario, individual users upload one image at a time.

What we need is fundamentally different: a way for the **server** to automatically combine individual requests that are waiting in the queue and process them as a batch. This is exactly what **dynamic batching** provides - and unlike client-side batching, it can actually reduce queue delay because it clears multiple requests from the queue in a single pass.

### Dynamic Batching

We have seen that batching dramatically improves throughput, but client-side batching does not reduce queue delay and is impractical for real users who send one image at a time.

**Dynamic batching** solves both problems. Instead of requiring the client to batch, the Triton server automatically groups individual requests that are waiting in the queue into a batch before sending them to the GPU. This is completely transparent to the client - each user still sends a single image, but the server combines them behind the scenes.

Unlike client-side batching, dynamic batching can actually **reduce queue delay** because it clears multiple requests from the queue in a single GPU pass.

#### Before dynamic batching

Before we enable dynamic batching, let us establish a reference point. Run the following tests with the current configuration (no dynamic batching), sending individual images (`-b 1`):

In [27]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 960
    Throughput: 53.1367 infer/sec
    Avg latency: 18745 usec (standard deviation 1005 usec)
    p50 latency: 18649 usec
    p90 latency: 19021 usec
    p95 latency: 19125 usec
    p99 latency: 19603 usec
    Avg HTTP time: 18740 usec (send/recv 171 usec + response wait 18569 usec)
  Server: 
    Inference count: 960
    Execution count: 960
    Successful request count: 960
    Avg request latency: 18137 usec (overhead 3 usec + queue 20 usec + compute input 53 usec + compute infer 18011 usec + compute output 49 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 53.1367 infer/sec, latency 18745 usec


In [28]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 1014
    Throughput: 56.1314 infer/sec
    Avg latency: 141903 usec (standard deviation 6746 usec)
    p50 latency: 141622 usec
    p90 latency: 146576 usec
    p95 latency: 146866 usec
    p99 latency: 147407 usec
    Avg HTTP time: 141897 usec (send/recv 180 usec + response wait 141717 usec)
  Server: 
    Inference count: 1014
    Execution count: 1014
    Successful request count: 1014
    Avg request latency: 141267 usec (overhead 2 usec + queue 123499 usec + compute input 57 usec + compute infer 17660 usec + compute output 49 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 56.1314 infer/sec, lat

#### Enabling dynamic batching

Now let us enable dynamic batching. Edit the model configuration file at `~/serve-system-chi/models/food_classifier/config.pbtxt`:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

The file currently looks like this:

    name: "food_classifier"
    backend: "python"
    max_batch_size: 16
    input [
      {
        name: "INPUT_IMAGE"
        data_type: TYPE_STRING
        dims: [1]
      }
    ]
    output [
      {
        name: "FOOD_LABEL"
        data_type: TYPE_STRING
        dims: [1]
      },
      {
        name: "PROBABILITY"
        data_type: TYPE_FP32
        dims: [1]
      }
    ]
      instance_group [
        {
          count: 1
          kind: KIND_GPU
          gpus: [ 0 ]
        }
    ]

At the end of the file, add:

    dynamic_batching {
      preferred_batch_size: [4, 6, 8, 10]
      max_queue_delay_microseconds: 100
    }

This tells Triton:

-   When there are requests waiting in the queue, try to form batches of size 4, 6, 8, or 10
-   Wait at most 100 microseconds for more requests to arrive before processing the current batch

Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

```bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

#### After dynamic batching

Let us get some pre-benchmark stats about how many requests have been served, broken down by batch size. (If you have just restarted the server, it would be zero\!)

In [29]:
# runs inside Jupyter container
curl -s http://triton_server:8000/v2/models/food_classifier/versions/1/stats | python3 -m json.tool

{
    "model_stats": [
        {
            "name": "food_classifier",
            "version": "1",
            "last_inference": 0,
            "inference_count": 0,
            "execution_count": 0,
            "inference_stats": {
                "success": {
                    "count": 0,
                    "ns": 0
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 0,
                    "ns": 0
                },
                "compute_input": {
                    "count": 0,
                    "ns": 0
                },
                "compute_infer": {
                    "count": 0,
                    "ns": 0
                },
                "compute_output": {
                    "count": 0,
                    "ns": 0
                },
                "cache_hit": {
                    "count": 0,
                    "ns": 0
             

Now run the same tests as before - first with concurrency 1, then concurrency 8:

In [35]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using asynchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 939
    Throughput: 51.9836 infer/sec
    Avg latency: 19111 usec (standard deviation 1103 usec)
    p50 latency: 18986 usec
    p90 latency: 19334 usec
    p95 latency: 19416 usec
    p99 latency: 20506 usec
    Avg HTTP time: 19081 usec (send/recv 227 usec + response wait 18854 usec)
  Server: 
    Inference count: 939
    Execution count: 939
    Successful request count: 939
    Avg request latency: 18440 usec (overhead 2 usec + queue 185 usec + compute input 53 usec + compute infer 18147 usec + compute output 52 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 51.9836 infer/sec, latency 19111 usec


In [36]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using asynchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 1552
    Throughput: 85.696 infer/sec
    Avg latency: 92867 usec (standard deviation 19140 usec)
    p50 latency: 82923 usec
    p90 latency: 131495 usec
    p95 latency: 137490 usec
    p99 latency: 138880 usec
    Avg HTTP time: 92837 usec (send/recv 283 usec + response wait 92554 usec)
  Server: 
    Inference count: 1552
    Execution count: 438
    Successful request count: 1552
    Avg request latency: 91550 usec (overhead 6 usec + queue 44073 usec + compute input 185 usec + compute infer 47131 usec + compute output 154 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 85.696 infer/sec, latency 92867 usec


Now get the per-batch stats again:

In [32]:
# runs inside Jupyter container
curl -s http://triton_server:8000/v2/models/food_classifier/versions/1/stats | python3 -m json.tool

{
    "model_stats": [
        {
            "name": "food_classifier",
            "version": "1",
            "last_inference": 1772510013364,
            "inference_count": 2486,
            "execution_count": 1354,
            "inference_stats": {
                "success": {
                    "count": 2486,
                    "ns": 161158413337
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 2486,
                    "ns": 65233738170
                },
                "compute_input": {
                    "count": 2486,
                    "ns": 327294211
                },
                "compute_infer": {
                    "count": 2486,
                    "ns": 95264276234
                },
                "compute_output": {
                    "count": 2486,
                    "ns": 322430536
                },
                "cache_h

You should observe:

-   **Concurrency 1**: little to no change - with only one client, there is nothing to batch. The server processes each request individually.
-   **Concurrency 8**: throughput improves and queue delay decreases. The server is now combining multiple waiting requests into a single batch, clearing the queue faster. We will also see a higher compute infer time because the execute() call will preprocess more images now which can add vey slight delay

Also look at the stats output from the `curl` command above. You should see `batch_stats` showing that requests were served in batch sizes greater than 1, even though each client sent a single image. The server is batching them automatically\!

#### Dynamic batching under different arrival patterns

Now let us revisit the constant vs. Poisson request patterns from Section 5, but this time with dynamic batching enabled. First, the reference without dynamic batching (from Section 5), then with it.

Run the following tests with dynamic batching still enabled:

In [57]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 50 --request-distribution constant

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using synchronous calls for inference

Request Rate: 50 inference requests per second
  Client: 
    Request count: 904
    Throughput: 49.9691 infer/sec
    Avg latency: 20753 usec (standard deviation 235 usec)
    p50 latency: 20711 usec
    p90 latency: 20923 usec
    p95 latency: 21035 usec
    p99 latency: 21869 usec
    Avg HTTP time: 20747 usec (send/recv 184 usec + response wait 20563 usec)
  Server: 
    Inference count: 904
    Execution count: 904
    Successful request count: 904
    Avg request latency: 20101 usec (overhead 2 usec + queue 241 usec + compute input 69 usec + compute infer 19730 usec + compute output 58 usec)
Inferences/Second vs. Client Average Batch

In [58]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 50 --request-distribution poisson

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 50 inference requests per second
  Client: 
    Request count: 910
    Avg send request rate: 50.43 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 19.12% of the requests were delayed. 
    Throughput: 50.43 infer/sec
    Avg latency: 39308 usec (standard deviation 15270 usec)
    p50 latency: 37571 usec
    p90 latency: 58138 usec
    p95 latency: 65261 usec
    p99 latency: 73462 usec
    Avg HTTP time: 39302 usec (send/recv 193 usec + response wait 39109 usec)
  Server: 
    Inference count: 910
    Execution count: 704
    Successful request count: 910
    Avg request latency: 

Now if we compare both the above results with no dynamic batching we can clearly observe a few things:
- We did not see any queueing delay in constant request rate of 50 req/s because the server was able to process the request as it came. But now with dynamic queueing enabled with the config where we have max_queue_delay_microseconds: 100, Triton server intentionally holds requests in the queue for up to 100μs, waiting for more requests to arrive so it can form a batch. Also the compute infer time increases because there might be smaller batches that are being created and hence preprocessing will be adding some sort of delay which could also result in queue builduop.
- for Poisson request rate, we will notice that the queueing delay has significantly dropped because server is batching requests and clearing up the queue faster. Due to this, there is an increase in the compute infer times but overall we see benefit. As poisson request as close to live traffic, we can see how dynamic batching can help. If we look at the `nvtop` when the test is running, we will see spikes in GPU usage indicating that server is likely processing a batch or batches of requests.

With dynamic batching enabled, let us also try a higher rate

In [59]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 70 --request-distribution poisson

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 70 inference requests per second
  Client: 
    Request count: 1208
    Avg send request rate: 66.95 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 77.73% of the requests were delayed. 
    Throughput: 66.73 infer/sec
    Avg latency: 52251 usec (standard deviation 10784 usec)
    p50 latency: 54673 usec
    p90 latency: 57418 usec
    p95 latency: 72654 usec
    p99 latency: 74008 usec
    Avg HTTP time: 52245 usec (send/recv 224 usec + response wait 52021 usec)
  Server: 
    Inference count: 1208
    Execution count: 704
    Successful request count: 1208
    Avg request latenc

Check the batch stats to see what batch sizes were formed under Poisson load:

In [60]:
# runs inside Jupyter container
curl -s http://triton_server:8000/v2/models/food_classifier/versions/1/stats | python3 -m json.tool

{
    "model_stats": [
        {
            "name": "food_classifier",
            "version": "1",
            "last_inference": 1772514531584,
            "inference_count": 18383,
            "execution_count": 13833,
            "inference_stats": {
                "success": {
                    "count": 18383,
                    "ns": 760197831446
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 18383,
                    "ns": 306730256548
                },
                "compute_input": {
                    "count": 18383,
                    "ns": 1394998516
                },
                "compute_infer": {
                    "count": 18383,
                    "ns": 450709859634
                },
                "compute_output": {
                    "count": 18383,
                    "ns": 1316510975
                },
             

You should observe that dynamic batching helps especially under **Poisson (bursty) arrivals**. In Section 5, burstiness was the problem - requests arriving in bursts caused queue buildup. Now with dynamic batching, those bursts are an **advantage**: when multiple requests arrive close together, the server combines them into a larger batch, processing them all in a single GPU pass.

The server can now handle 70 requests per second with Poisson arrivals. The very burstiness that was causing the problem is now helping form bigger, more efficient batches.

However, dynamic batching with the Python backend has limits. The `preferred_batch_size` and `max_queue_delay_microseconds` settings are just hints — they control how long the server waits and what batch sizes it prefers, but the actual batch sizes depend on how many requests are in the queue at any moment. With this backend, the Python preprocessing overhead (base64 decoding, image resizing, normalization) runs sequentially for each image in the `execute()` method, which limits how much throughput improvement we can get from batching alone. We need to either run more instances of the model, or switch to a more efficient backend.

### Effect of Maximum Batch Size on Dynamic Batching

We have seen that dynamic batching helps under load, but how much does the **maximum batch size** matter? The `max_batch_size` parameter in `config.pbtxt` sets a hard cap on how many requests the server can combine into a single batch. A larger cap allows the server to clear more requests per GPU pass; a smaller cap forces the server to process them in more, smaller batches.

Let us compare two configurations under Poisson arrivals to see this tradeoff.

#### Configuration 1: Large maximum batch size

Edit `~/serve-system-chi/models/food_classifier/config.pbtxt`:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

Change `max_batch_size` and the `dynamic_batching` block to:

    max_batch_size: 16

    dynamic_batching {
      preferred_batch_size: [8, 12, 16]
      max_queue_delay_microseconds: 100
    }

This allows Triton to batch up to 16 requests in a single GPU pass. Save the file, then rebuild and restart:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

Wait for the server to come back up (`docker logs triton_server`), then run:

In [61]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 50 --request-distribution poisson --async 

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 50 inference requests per second
  Client: 
    Request count: 911
    Throughput: 50.4654 infer/sec
    Avg latency: 41432 usec (standard deviation 20028 usec)
    p50 latency: 36879 usec
    p90 latency: 69060 usec
    p95 latency: 81449 usec
    p99 latency: 100867 usec
    Avg HTTP time: 41408 usec (send/recv 200 usec + response wait 41208 usec)
  Server: 
    Inference count: 911
    Execution count: 688
    Successful request count: 911
    Avg request latency: 40741 usec (overhead 3 usec + queue 16545 usec + compute input 86 usec + compute infer 24033 usec + compute output 73 usec)
Inferences/Second vs. Client Average Batch Laten

In [76]:
# runs inside Jupyter container
curl -s http://triton_server:8000/v2/models/food_classifier/versions/1/stats | python3 -m json.tool

{
    "model_stats": [
        {
            "name": "food_classifier",
            "version": "1",
            "last_inference": 1772515562025,
            "inference_count": 1466,
            "execution_count": 480,
            "inference_stats": {
                "success": {
                    "count": 1466,
                    "ns": 143774367423
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 1466,
                    "ns": 81730766992
                },
                "compute_input": {
                    "count": 1466,
                    "ns": 202674106
                },
                "compute_infer": {
                    "count": 1466,
                    "ns": 61643462261
                },
                "compute_output": {
                    "count": 1466,
                    "ns": 190979398
                },
                "cache_hi

In [75]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8 --async

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using asynchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 1456
    Throughput: 80.4716 infer/sec
    Avg latency: 99018 usec (standard deviation 25970 usec)
    p50 latency: 109559 usec
    p90 latency: 112115 usec
    p95 latency: 118887 usec
    p99 latency: 139738 usec
    Avg HTTP time: 98993 usec (send/recv 282 usec + response wait 98711 usec)
  Server: 
    Inference count: 1456
    Execution count: 476
    Successful request count: 1456
    Avg request latency: 98100 usec (overhead 5 usec + queue 55757 usec + compute input 138 usec + compute infer 42069 usec + compute output 130 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 80.4716 infer/sec, latency 99018 use

Note the batch sizes in the stats output. With `max_batch_size: 16`, the server can grab as many queued requests as available (up to 16) in a single pass.

#### Configuration 2: Small maximum batch size

Now let us hard-cap the batch size to 2. Edit `~/serve-system-chi/models/food_classifier/config.pbtxt`:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

Change `max_batch_size` and the `dynamic_batching` block to:

    max_batch_size: 2

    dynamic_batching {
      preferred_batch_size: [2]
      max_queue_delay_microseconds: 100
    }

This hard-caps each batch to at most 2 requests. Even if 10 requests are waiting in the queue, the server can only grab 2 at a time. Save, rebuild, and restart:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

Wait for the server to come back up, then run the same Poisson 50 req/sec benchmark:

In [72]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --request-rate-range 50 --request-distribution poisson --async 

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 50 inference requests per second
  Client: 
    Request count: 910
    Throughput: 50.4084 infer/sec
    Avg latency: 41140 usec (standard deviation 20393 usec)
    p50 latency: 36724 usec
    p90 latency: 69003 usec
    p95 latency: 82063 usec
    p99 latency: 100043 usec
    Avg HTTP time: 41119 usec (send/recv 199 usec + response wait 40920 usec)
  Server: 
    Inference count: 910
    Execution count: 703
    Successful request count: 910
    Avg request latency: 40450 usec (overhead 2 usec + queue 18415 usec + compute input 71 usec + compute infer 21896 usec + compute output 65 usec)
Inferences/Second vs. Client Average Batch Laten

In [73]:
# runs inside Jupyter container
curl -s http://triton_server:8000/v2/models/food_classifier/versions/1/stats | python3 -m json.tool

{
    "model_stats": [
        {
            "name": "food_classifier",
            "version": "1",
            "last_inference": 1772515050076,
            "inference_count": 2060,
            "execution_count": 1589,
            "inference_stats": {
                "success": {
                    "count": 2060,
                    "ns": 89313549468
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 2060,
                    "ns": 43004122425
                },
                "compute_input": {
                    "count": 2060,
                    "ns": 143942380
                },
                "compute_infer": {
                    "count": 2060,
                    "ns": 46024963885
                },
                "compute_output": {
                    "count": 2060,
                    "ns": 135778781
                },
                "cache_hi

In [77]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8 --async

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using asynchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 1340
    Throughput: 74.0215 infer/sec
    Avg latency: 107990 usec (standard deviation 2880 usec)
    p50 latency: 106342 usec
    p90 latency: 111482 usec
    p95 latency: 111690 usec
    p99 latency: 112787 usec
    Avg HTTP time: 107962 usec (send/recv 252 usec + response wait 107710 usec)
  Server: 
    Inference count: 1338
    Execution count: 669
    Successful request count: 1338
    Avg request latency: 107139 usec (overhead 3 usec + queue 80209 usec + compute input 113 usec + compute infer 26725 usec + compute output 88 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 74.0215 infer/sec, latency 107990 

In case of these tests with poisson distribution we cannot really see much of a difference. We can see that in case of higher batch size, queueing delay is a biut lower but the compute infer is slightly higher compared to smaller batch size. To see actual tradeoffs and effects we will shift to concurrent-range just for this comparison

In case of the concurrent tests that we did we can clearly see the following:
Large batches → higher compute_infer (processing more images at once) but much lower queue delay because requests get drained faster. Net result: better throughput and lower latency.
Small batches → lower compute_infer per execution, but requests pile up in the queue waiting because batches are capped at 2.


#### Reset configuration

Before moving on, reset the configuration back to the original settings. Edit `~/serve-system-chi/models/food_classifier/config.pbtxt`:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

Change back to:

    max_batch_size: 16

    dynamic_batching {
      preferred_batch_size: [4, 6, 8, 10]
      max_queue_delay_microseconds: 100
    }

Save, rebuild, and restart:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

Even with the best batching configuration, we are hitting a ceiling with the Python backend. The preprocessing overhead in `model.py` runs sequentially in Python for each image, which limits how much throughput we can gain from batching alone. We need to either run more instances of the model, or switch to a more efficient backend.

### Scaling up

Another easy way to improve performance is to scale up! Let's edit the model configuration:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

and change

    instance_group [
      {
        count: 1
        kind: KIND_GPU
        gpus: [ 0 ]
      }
    ]

to run two instances on GPU 0 and two instances on GPU 1:

    instance_group [
      {
        count: 2
        kind: KIND_GPU
        gpus: [ 0 ]
      },
      {
        count: 2
        kind: KIND_GPU
        gpus: [ 1 ]
      }
    ]

Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

```bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

On the host, run

```bash
# runs on node-serve-system
nvidia-smi
```

and note that there are two instances of `triton_python_backend` processes running on GPU 0, and two on GPU 1.

Then, benchmark *this* service with increased concurrency:

In [78]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 3573
    Throughput: 196.097 infer/sec
    Avg latency: 40683 usec (standard deviation 19032 usec)
    p50 latency: 36858 usec
    p90 latency: 57174 usec
    p95 latency: 69475 usec
    p99 latency: 88897 usec
    Avg HTTP time: 40677 usec (send/recv 146 usec + response wait 40531 usec)
  Server: 
    Inference count: 3573
    Execution count: 2393
    Successful request count: 3573
    Avg request latency: 40117 usec (overhead 3 usec + queue 6653 usec + compute input 70 usec + compute infer 33320 usec + compute output 69 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 196.097 infer/sec, latency 4068

Although there is still some queuing delay (because our degree of concurrency, 8, is still higher than the number of server instances, 4), and the inference time is also increased due to sharing the compute resources, the prediction delay is still on the order of 10s of ms - not over 100ms, like it was previously with concurrency 8!

Also, if you look at the `nvtop` output on the host while running this test, you will observe higher GPU utilization than before (which is good! We want to use the GPU. Underutilization is bad.) (Take a screenshot!) However, we are still not fully utilizing the GPU.

Let's try increasing the number of instances again. Edit the model configuration:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

and change

    instance_group [
      {
        count: 2
        kind: KIND_GPU
        gpus: [ 0 ]
      },
      {
        count: 2
        kind: KIND_GPU
        gpus: [ 1 ]
      }
    ]

to

    instance_group [
      {
        count: 4
        kind: KIND_GPU
        gpus: [ 0 ]
      },
      {
        count: 4
        kind: KIND_GPU
        gpus: [ 1 ]
      }
    ]

Re-build the container image with this change:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

use

```bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Then, re-run our benchmark:

In [80]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier --input-data input.json -b 1 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 2209
    Throughput: 122.122 infer/sec
    Avg latency: 65430 usec (standard deviation 38787 usec)
    p50 latency: 55393 usec
    p90 latency: 115947 usec
    p95 latency: 135853 usec
    p99 latency: 169641 usec
    Avg HTTP time: 65413 usec (send/recv 149 usec + response wait 65264 usec)
  Server: 
    Inference count: 2210
    Execution count: 1730
    Successful request count: 2210
    Avg request latency: 64710 usec (overhead 3 usec + queue 486 usec + compute input 60 usec + compute infer 64072 usec + compute output 89 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 122.122 infer/sec, latency 65

This makes things worse - our inference time is higher, even though we are still underutilizing the GPU (as seen in `nvtop`) (take a screenshot!).

Our system is not limited by GPU - we are underutilizing the GPU. However, we are being killed by the overhead of the Python backend and our `model.py` implementation.

### Serving an ONNX model

The Python backend we have been using is flexible, but not necessarily the most performant. To get better performance, we will use one of the highly optimized backends in Triton. Since we already have an ONNX model, let's use the ONNX backend.

To serve a model using the ONNX backend, we will create a [directory structure like this](https://github.com/teaching-on-testbeds/serve-system-chi/tree/main/models_staging/food_classifier_onnx):

    food_classifier_onnx/
    ├── 1
    │   └── model.onnx
    └── config.pbtxt

There is no more `model.py` - Triton serves the model directly, we just have to name it `model.onnx`. In [`config.pbtxt`](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/models_staging/food_classifier_onnx/config.pbtxt), we will specify the backend as `onnxruntime`:

    name: "food_classifier_onnx"
    backend: "onnxruntime"
    max_batch_size: 16
    input [
      {
        name: "input"  # has to match ONNX model's input name
        data_type: TYPE_FP32
        dims: [3, 224, 224]  # has to match ONNX input shape
      }
    ]
    output [
      {
        name: "output"  # has to match ONNX model output name
        data_type: TYPE_FP32  # output is a list of probabilities
        dims: [11]
      }
    ]
      instance_group [
        {
          count: 1
          kind: KIND_GPU
          gpus: [ 0 ]
        }
    ]

Copy this to Triton's models directory:

```bash
# runs on node-serve-system
cp -r ~/serve-system-chi/models_staging/food_classifier_onnx ~/serve-system-chi/models/
```

Re-build the container image with this change:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

use

```bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready. Note that the server will load two models: the original `food_classifier` with Python backend, and the `food_classifier_onnx` model we just added.

Let's benchmark our service. Our ONNX model won't accept image bytes directly - it expects images that already have been pre-processed into arrays. So, our benchmark command will be a little bit different:

In [82]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 2577
    Throughput: 132.94 infer/sec
    Avg latency: 6846 usec (standard deviation 1192 usec)
    p50 latency: 6477 usec
    p90 latency: 8512 usec
    p95 latency: 9461 usec
    p99 latency: 9985 usec
    Avg HTTP time: 6841 usec (send/recv 761 usec + response wait 6080 usec)
  Server: 
    Inference count: 2577
    Execution count: 2577
    Successful request count: 2577
    Avg request latency: 4665 usec (overhead 25 usec + queue 25 usec + compute input 101 usec + compute infer 4498 usec + compute output 15 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 132.94 infer/sec, latency 6846 usec


This model has much better inference performance than our PyTorch model with Python backend did, in a similar test. Also, if we monitor with `nvtop`, we should see higher GPU utilization while the test is running (which is a good thing!) (Take a screenshot!)

Let's try scaling *this* model up. Edit the model configuration:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier_onnx/config.pbtxt
```

and change

    instance_group [
      {
        count: 1
        kind: KIND_GPU
        gpus: [ 0 ]
      }
    ]

to

    instance_group [
      {
        count: 2
        kind: KIND_GPU
        gpus: [ 0, 1 ]
      }
    ]

Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

```bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Then, run our benchmark with higher concurrency. (2 instances on each GPU, because we noticed that a single instance used less than half a GPU.)

Watch the `nvtop` output as you run this test! (Take a screenshot!)

In [83]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --concurrency-range 8

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 100815
    Throughput: 1171.93 infer/sec
    Avg latency: 6071 usec (standard deviation 830 usec)
    p50 latency: 5979 usec
    p90 latency: 6871 usec
    p95 latency: 7222 usec
    p99 latency: 8909 usec
    Avg HTTP time: 6064 usec (send/recv 627 usec + response wait 5437 usec)
  Server: 
    Inference count: 100814
    Execution count: 100814
    Successful request count: 100814
    Avg request latency: 4039 usec (overhead 19 usec + queue 754 usec + compute input 109 usec + compute infer 3144 usec + compute output 11 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 1171.93 infer/sec, latency 6071 usec


This time, we should see that our model is fully utilizing the GPU (that's good!) And, our inference performance is much better than the PyTorch model with Python backend could achieve with concurrency 8.

Let's see how we do with even higher concurrency. Here we are shifting to gRPC. HTTP creates a separate TCP connection per request, so 16+ concurrent requests exhaust the server's thread pool and connection limits — gRPC multiplexes all requests over a single TCP connection, handling high concurrency without overwhelming the server.

In [4]:
# runs inside Jupyter container
# perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --concurrency-range 16
# perf_analyzer -u triton_server:8001 -i grpc -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --concurrency-range 16 --measurement-interval 10000
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --concurrency-range 16 --warmup-request-count 100 --request-count 10000

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 warmup requests
  Sending 10000 benchmark requests
  Using synchronous calls for inference

Request concurrency: 16
Request concurrency: 16
  Client: 
    Request count: 10000
    Throughput: 1110.7 infer/sec
    Avg latency: 12768 usec (standard deviation 812 usec)
    p50 latency: 12675 usec
    p90 latency: 13409 usec
    p95 latency: 13668 usec
    p99 latency: 14433 usec
    Avg HTTP time: 12762 usec (send/recv 906 usec + response wait 11856 usec)
  Server: 
    Inference count: 10000
    Execution count: 10000
    Successful request count: 10000
    Avg request latency: 10396 usec (overhead 19 usec + queue 7084 usec + compute input 112 usec + compute infer 3169 usec + compute output 11 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 16, throughput: 1110.7 infer/sec, latency 12768 usec


We still have some queue delay, since the rate at which requests arrive is greater than the service rate of the models. But, we can feel good that we are no longer underutilizing the GPUs!

There's one more issue we should address: our ONNX model doesn't directly work with our Flask server now, because the inputs and outputs are different. The ONNX model expects a pre-processed array, and returns a list of class probabilities.

Since the pre-processing and post-processing doesn't need GPU anyway, we'll move it to the Flask app.

Edit the Docker compose file:

```bash
# runs on node-serve-system
nano ~/serve-system-chi/docker/docker-compose-triton.yaml
```

and change

    flask:
      build:
        context: https://github.com/teaching-on-testbeds/gourmetgram.git#triton

to

    flask:
      build:
        context: https://github.com/teaching-on-testbeds/gourmetgram.git#triton_onnx

to use [a version of our Flask app where the pre- and post-processing is built in](https://github.com/teaching-on-testbeds/gourmetgram/blob/triton_onnx/app.py). Also change

      - FOOD11_MODEL_NAME=food_classifier

to

      - FOOD11_MODEL_NAME=food_classifier_onnx

so that our Flask app will send requests to the new ONNX model service.

Then run

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build flask
```

to re-build the container image, and

```bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up flask --force-recreate -d
```

to restart the Flask container with the new image.

Let's test this service. In a browser, run

    http://A.B.C.D

but substitute the floating IP assigned to your instance, to access the Flask app. Upload an image and press "Submit" to get its class label.

Then, download this entire notebook for later reference.